Name - Nikhil

Roll No. - 8025320062


In [129]:
!pip install mrjob

In [130]:
from collections import defaultdict, Counter
import re
import os

In [131]:
def run_manual_mapreduce(input_data, mapper, reducer):
    """
    input_data: iterable of records
    mapper: function(record) -> list of (key, value)
    reducer: function(key, list_of_values) -> final_value
    """
    mapped = []
    for record in input_data:
        mapped.extend(mapper(record))

    shuffled = defaultdict(list)
    for key, value in mapped:
        shuffled[key].append(value)

    reduced = {}
    for key, values in shuffled.items():
        reduced[key] = reducer(key, values)

    return mapped, dict(shuffled), reduced

Q1 Word Count - Count the frequency of each word

Input:

• hadoop is fast

• hadoop is scalable

(A) Manual Python Approach

In [132]:
q1_lines = [
    "hadoop is fast",
    "hadoop is scalable"
]

def q1_mapper(line):
    return [(word.lower(), 1) for word in line.split()]

def q1_reducer(word, counts):
    return sum(counts)

mapped, shuffled, reduced = run_manual_mapreduce(q1_lines, q1_mapper, q1_reducer)

print("Mapped:", mapped)
print("Shuffled:", shuffled)
print("Reduced:", reduced)

Mapped: [('hadoop', 1), ('is', 1), ('fast', 1), ('hadoop', 1), ('is', 1), ('scalable', 1)]
Shuffled: {'hadoop': [1, 1], 'is': [1, 1], 'fast': [1], 'scalable': [1]}
Reduced: {'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


(B) MRJob Framework Approach

In [133]:
%%writefile q1_word_count_mrjob.py
from mrjob.job import MRJob

class WordCountMR(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word.lower(), 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    WordCountMR.run()

Overwriting q1_word_count_mrjob.py


In [134]:
with open("q1_input.txt", "w") as f:
    f.write("hadoop is fast\n")
    f.write("hadoop is scalable\n")

!python q1_word_count_mrjob.py q1_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q1_word_count_mrjob.root.20260507.081147.915886
Running step 1 of 1...
job output is in /tmp/q1_word_count_mrjob.root.20260507.081147.915886/output
Streaming final output from /tmp/q1_word_count_mrjob.root.20260507.081147.915886/output...
"fast"	1
"hadoop"	2
"scalable"	1
"is"	2
Removing temp directory /tmp/q1_word_count_mrjob.root.20260507.081147.915886...


Q2 Character Count - Count the frequency of each character (ignore spaces).

Input: big data

(A) Manual Python Approach

In [135]:
q2_text = "big data"

def q2_mapper(text):
    return [(ch, 1) for ch in text if ch != " "]

def q2_reducer(ch, counts):
    return sum(counts)

mapped, shuffled, reduced = run_manual_mapreduce([q2_text], q2_mapper, q2_reducer)

print("Mapped:", mapped)
print("Shuffled:", shuffled)
print("Reduced:", reduced)

Mapped: [('b', 1), ('i', 1), ('g', 1), ('d', 1), ('a', 1), ('t', 1), ('a', 1)]
Shuffled: {'b': [1], 'i': [1], 'g': [1], 'd': [1], 'a': [1, 1], 't': [1]}
Reduced: {'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


(B) MRJob Framework Approach

In [136]:
%%writefile q2_character_count_mrjob.py
from mrjob.job import MRJob

class CharacterCountMR(MRJob):
    def mapper(self, _, line):
        for ch in line:
            if ch != " ":
                yield ch, 1

    def reducer(self, ch, counts):
        yield ch, sum(counts)

if __name__ == "__main__":
    CharacterCountMR.run()

Overwriting q2_character_count_mrjob.py


In [137]:
with open("q2_input.txt", "w") as f:
    f.write("big data\n")

!python q2_character_count_mrjob.py q2_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q2_character_count_mrjob.root.20260507.081148.358216
Running step 1 of 1...
job output is in /tmp/q2_character_count_mrjob.root.20260507.081148.358216/output
Streaming final output from /tmp/q2_character_count_mrjob.root.20260507.081148.358216/output...
"a"	2
"g"	1
"i"	1
"b"	1
"d"	1
"t"	1
Removing temp directory /tmp/q2_character_count_mrjob.root.20260507.081148.358216...


Q3 Average Word Length (Per Word) - Compute the average length of each word.

Input: data science data big data

(A) Manual Python Approach

In [138]:
q3_text = "data science data big data"

def q3_mapper(text):
    return [(word.lower(), len(word)) for word in text.split()]

def q3_reducer(word, lengths):
    return sum(lengths) / len(lengths)

mapped, shuffled, reduced = run_manual_mapreduce([q3_text], q3_mapper, q3_reducer)

print("Mapped:", mapped)
print("Shuffled:", shuffled)
print("Reduced:", reduced)

Mapped: [('data', 4), ('science', 7), ('data', 4), ('big', 3), ('data', 4)]
Shuffled: {'data': [4, 4, 4], 'science': [7], 'big': [3]}
Reduced: {'data': 4.0, 'science': 7.0, 'big': 3.0}


(B) MRJob Framework Approach

In [139]:
%%writefile q3_avg_word_length_per_word_mrjob.py
from mrjob.job import MRJob

class AvgWordLengthPerWordMR(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word.lower(), len(word)

    def reducer(self, word, lengths):
        lengths = list(lengths)
        yield word, sum(lengths) / len(lengths)

if __name__ == "__main__":
    AvgWordLengthPerWordMR.run()

Overwriting q3_avg_word_length_per_word_mrjob.py


In [140]:
with open("q3_input.txt", "w") as f:
    f.write("data science data big data\n")

!python q3_avg_word_length_per_word_mrjob.py q3_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q3_avg_word_length_per_word_mrjob.root.20260507.081148.690108
Running step 1 of 1...
job output is in /tmp/q3_avg_word_length_per_word_mrjob.root.20260507.081148.690108/output
Streaming final output from /tmp/q3_avg_word_length_per_word_mrjob.root.20260507.081148.690108/output...
"big"	3.0
"data"	4.0
"science"	7.0
Removing temp directory /tmp/q3_avg_word_length_per_word_mrjob.root.20260507.081148.690108...


Q4 Global Average Word Length - Compute the average length of all words.

Input: hadoop mapreduce spark

(A) Manual Python Approach

In [141]:
q4_text = "hadoop mapreduce spark"

def q4_mapper(text):
    return [("global", (len(word), 1)) for word in text.split()]

def q4_reducer(key, values):
    total_length = sum(length for length, count in values)
    total_count = sum(count for length, count in values)
    return total_length / total_count

mapped, shuffled, reduced = run_manual_mapreduce([q4_text], q4_mapper, q4_reducer)

print("Mapped:", mapped)
print("Shuffled:", shuffled)
print("Reduced:", reduced)

Mapped: [('global', (6, 1)), ('global', (9, 1)), ('global', (5, 1))]
Shuffled: {'global': [(6, 1), (9, 1), (5, 1)]}
Reduced: {'global': 6.666666666666667}


(B) MRJob Framework Approach

In [142]:
%%writefile q4_global_avg_word_length_mrjob.py
from mrjob.job import MRJob

class GlobalAvgWordLengthMR(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield "Global Average", (len(word), 1)

    def reducer(self, key, values):
        total_length = 0
        total_count = 0
        for length, count in values:
            total_length += length
            total_count += count
        yield key, total_length / total_count

if __name__ == "__main__":
    GlobalAvgWordLengthMR.run()

Overwriting q4_global_avg_word_length_mrjob.py


In [143]:
with open("q4_input.txt", "w") as f:
    f.write("hadoop mapreduce spark\n")

!python q4_global_avg_word_length_mrjob.py q4_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q4_global_avg_word_length_mrjob.root.20260507.081149.168325
Running step 1 of 1...
job output is in /tmp/q4_global_avg_word_length_mrjob.root.20260507.081149.168325/output
Streaming final output from /tmp/q4_global_avg_word_length_mrjob.root.20260507.081149.168325/output...
"Global Average"	6.666666666666667
Removing temp directory /tmp/q4_global_avg_word_length_mrjob.root.20260507.081149.168325...


Perform Q1-Q4 on the file available on the link: https://drive.google.com/file/d/16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas/view?usp=sharing

• Also find Top 5 most frequent words

From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.

(A) Manual Python Approach

In [145]:
#Word Count
import re
data = ["""From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee."""]

mapped = []

# MAPPER
for line in data:
    words = re.findall(r'\w+', line.lower())
    for word in words:
        mapped.append((word, 1))

# SHUFFLE
shuffle = {}
for word, count in mapped:
    shuffle.setdefault(word, []).append(count)

# REDUCER
reduced = {}
for word, counts in shuffle.items():
    reduced[word] = sum(counts)

print(reduced)

{'from': 1, 'fairest': 1, 'creatures': 1, 'we': 1, 'desire': 1, 'increase': 1, 'that': 2, 'thereby': 1, 'beauty': 1, 's': 4, 'rose': 1, 'might': 2, 'never': 1, 'die': 1, 'but': 2, 'as': 1, 'the': 6, 'riper': 1, 'should': 1, 'by': 2, 'time': 1, 'decease': 1, 'his': 2, 'tender': 2, 'heir': 1, 'bear': 1, 'memory': 1, 'thou': 2, 'contracted': 1, 'to': 4, 'thine': 2, 'own': 2, 'bright': 1, 'eyes': 1, 'feed': 1, 'st': 2, 'thy': 5, 'light': 1, 'flame': 1, 'with': 1, 'self': 3, 'substantial': 1, 'fuel': 1, 'making': 1, 'a': 1, 'famine': 1, 'where': 1, 'abundance': 1, 'lies': 1, 'foe': 1, 'sweet': 1, 'too': 1, 'cruel': 1, 'art': 1, 'now': 1, 'world': 3, 'fresh': 1, 'ornament': 1, 'and': 3, 'only': 1, 'herald': 1, 'gaudy': 1, 'spring': 1, 'within': 1, 'bud': 1, 'buriest': 1, 'content': 1, 'churl': 1, 'mak': 1, 'waste': 1, 'in': 1, 'niggarding': 1, 'pity': 1, 'or': 1, 'else': 1, 'this': 1, 'glutton': 1, 'be': 1, 'eat': 1, 'due': 1, 'grave': 1, 'thee': 1}


In [146]:
#Character Count
mapped = []

for line in data:
    for ch in line.lower():
        if ch != " " and ch.isalnum():
            mapped.append((ch, 1))

shuffle = {}
for ch, count in mapped:
    shuffle.setdefault(ch, []).append(count)

reduced = {ch: sum(v) for ch, v in shuffle.items()}

print(reduced)

{'f': 11, 'r': 33, 'o': 25, 'm': 11, 'a': 30, 'i': 29, 'e': 68, 's': 30, 't': 58, 'c': 9, 'u': 17, 'w': 12, 'd': 20, 'n': 29, 'h': 34, 'b': 13, 'y': 14, 'g': 12, 'v': 2, 'p': 3, 'l': 18, 'k': 2}


In [147]:
#Average Word Length
mapped = []

for line in data:
    words = re.findall(r'\w+', line.lower())
    for word in words:
        mapped.append((word, len(word)))

shuffle = {}
for word, length in mapped:
    shuffle.setdefault(word, []).append(length)

reduced = {}
for word, lengths in shuffle.items():
    reduced[word] = sum(lengths) / len(lengths)

print(reduced)

{'from': 4.0, 'fairest': 7.0, 'creatures': 9.0, 'we': 2.0, 'desire': 6.0, 'increase': 8.0, 'that': 4.0, 'thereby': 7.0, 'beauty': 6.0, 's': 1.0, 'rose': 4.0, 'might': 5.0, 'never': 5.0, 'die': 3.0, 'but': 3.0, 'as': 2.0, 'the': 3.0, 'riper': 5.0, 'should': 6.0, 'by': 2.0, 'time': 4.0, 'decease': 7.0, 'his': 3.0, 'tender': 6.0, 'heir': 4.0, 'bear': 4.0, 'memory': 6.0, 'thou': 4.0, 'contracted': 10.0, 'to': 2.0, 'thine': 5.0, 'own': 3.0, 'bright': 6.0, 'eyes': 4.0, 'feed': 4.0, 'st': 2.0, 'thy': 3.0, 'light': 5.0, 'flame': 5.0, 'with': 4.0, 'self': 4.0, 'substantial': 11.0, 'fuel': 4.0, 'making': 6.0, 'a': 1.0, 'famine': 6.0, 'where': 5.0, 'abundance': 9.0, 'lies': 4.0, 'foe': 3.0, 'sweet': 5.0, 'too': 3.0, 'cruel': 5.0, 'art': 3.0, 'now': 3.0, 'world': 5.0, 'fresh': 5.0, 'ornament': 8.0, 'and': 3.0, 'only': 4.0, 'herald': 6.0, 'gaudy': 5.0, 'spring': 6.0, 'within': 6.0, 'bud': 3.0, 'buriest': 7.0, 'content': 7.0, 'churl': 5.0, 'mak': 3.0, 'waste': 5.0, 'in': 2.0, 'niggarding': 10.0, 'pi

In [148]:
#Global Average Word Length
total_len = 0
total_count = 0

for line in data:
    words = re.findall(r'\w+', line.lower())
    for word in words:
        total_len += len(word)
        total_count += 1

print(total_len / total_count)

4.247787610619469


In [149]:
#Top 5 Most Frequent Words
top5 = sorted(reduced.items(), key=lambda x: x[1], reverse=True)[:5]
print(top5)

[('substantial', 11.0), ('contracted', 10.0), ('niggarding', 10.0), ('creatures', 9.0), ('abundance', 9.0)]


(B) MRJob Framework Approach

In [150]:
poem = """From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thy self thy foe, to thy sweet self too cruel:
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
Within thine own bud buriest thy content,
And tender churl mak'st waste in niggarding:
Pity the world, or else this glutton be,
To eat the world's due, by the grave and thee."""

with open("sonnet.txt", "w", encoding="utf-8") as f:
    f.write(poem)

In [151]:
#Word Count
%%writefile word_count_mrjob.py
from mrjob.job import MRJob
import re

WORD_RE = re.compile(r"[A-Za-z']+")

class MRWordCount(MRJob):

    def mapper(self, _, line):
        words = WORD_RE.findall(line.lower())
        for word in words:
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordCount.run()

Overwriting word_count_mrjob.py


In [152]:
!python word_count_mrjob.py sonnet.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/word_count_mrjob.root.20260507.081149.855604
Running step 1 of 1...
job output is in /tmp/word_count_mrjob.root.20260507.081149.855604/output
Streaming final output from /tmp/word_count_mrjob.root.20260507.081149.855604/output...
"a"	1
"abundance"	1
"and"	3
"art"	1
"as"	1
"be"	1
"bear"	1
"beauty's"	1
"bright"	1
"bud"	1
"buriest"	1
"but"	2
"by"	2
"churl"	1
"content"	1
"contracted"	1
"creatures"	1
"cruel"	1
"decease"	1
"desire"	1
"die"	1
"due"	1
"eat"	1
"never"	1
"niggarding"	1
"now"	1
"only"	1
"or"	1
"ornament"	1
"own"	2
"pity"	1
"riper"	1
"rose"	1
"self"	3
"should"	1
"spring"	1
"substantial"	1
"sweet"	1
"tender"	2
"that"	2
"the"	6
"else"	1
"eyes"	1
"fairest"	1
"famine"	1
"feed'st"	1
"flame"	1
"foe"	1
"fresh"	1
"from"	1
"fuel"	1
"gaudy"	1
"glutton"	1
"grave"	1
"heir"	1
"herald"	1
"his"	2
"in"	1
"increase"	1
"lies"	1
"light's"	1
"mak'st"	1
"making"	1
"memory"	1
"might"

In [153]:
#Character Count
%%writefile charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):

    def mapper(self, _, line):
        for ch in line.lower():
            if ch != " " and ch.isalnum():
                yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Overwriting charcount.py


In [154]:
!python charcount.py sonnet.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charcount.root.20260507.081150.463074
Running step 1 of 1...
job output is in /tmp/charcount.root.20260507.081150.463074/output
Streaming final output from /tmp/charcount.root.20260507.081150.463074/output...
"a"	30
"b"	13
"c"	9
"d"	20
"e"	68
"o"	25
"p"	3
"r"	33
"s"	30
"t"	58
"f"	11
"g"	12
"h"	34
"i"	29
"k"	2
"l"	18
"m"	11
"n"	29
"u"	17
"v"	2
"w"	12
"y"	14
Removing temp directory /tmp/charcount.root.20260507.081150.463074...


In [155]:
#Average Word Length
%%writefile avg_word_len.py
from mrjob.job import MRJob
import re

class AvgWordLen(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, len(word)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    AvgWordLen.run()

Overwriting avg_word_len.py


In [156]:
!python avg_word_len.py sonnet.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_word_len.root.20260507.081151.118428
Running step 1 of 1...
job output is in /tmp/avg_word_len.root.20260507.081151.118428/output
Streaming final output from /tmp/avg_word_len.root.20260507.081151.118428/output...
"a"	1.0
"abundance"	9.0
"and"	3.0
"art"	3.0
"as"	2.0
"be"	2.0
"bear"	4.0
"beauty"	6.0
"bright"	6.0
"bud"	3.0
"buriest"	7.0
"but"	3.0
"by"	2.0
"churl"	5.0
"content"	7.0
"contracted"	10.0
"creatures"	9.0
"cruel"	5.0
"decease"	7.0
"desire"	6.0
"die"	3.0
"due"	3.0
"eat"	3.0
"else"	4.0
"now"	3.0
"only"	4.0
"or"	2.0
"ornament"	8.0
"own"	3.0
"pity"	4.0
"riper"	5.0
"rose"	4.0
"s"	1.0
"self"	4.0
"should"	6.0
"spring"	6.0
"st"	2.0
"substantial"	11.0
"sweet"	5.0
"tender"	6.0
"that"	4.0
"the"	3.0
"eyes"	4.0
"fairest"	7.0
"famine"	6.0
"feed"	4.0
"flame"	5.0
"foe"	3.0
"fresh"	5.0
"from"	4.0
"fuel"	4.0
"gaudy"	5.0
"glutton"	7.0
"grave"	5.0
"heir"	4.0
"herald"	6.0
"his

In [157]:
#Global Average Word Length
%%writefile global_avg.py
from mrjob.job import MRJob
import re

class GlobalAvg(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield "all", (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0

        for l, c in values:
            total_len += l
            total_count += c

        yield "Global Average", total_len / total_count

if __name__ == "__main__":
    GlobalAvg.run()

Overwriting global_avg.py


In [158]:
!python global_avg.py sonnet.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/global_avg.root.20260507.081151.939894
Running step 1 of 1...
job output is in /tmp/global_avg.root.20260507.081151.939894/output
Streaming final output from /tmp/global_avg.root.20260507.081151.939894/output...
"Global Average"	4.247787610619469
Removing temp directory /tmp/global_avg.root.20260507.081151.939894...


In [159]:
#Top 5 Most Frequent Words
%%file top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

class Top5Words(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_count),
            MRStep(reducer=self.reducer_top5)
        ]

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, 1

    def reducer_count(self, key, values):
        yield None, (sum(values), key)

    def reducer_top5(self, _, values):
        top5 = sorted(values, reverse=True)[:5]
        for count, word in top5:
            yield word, count

if __name__ == "__main__":
    Top5Words.run()

Overwriting top5.py


In [160]:
!python top5.py sonnet.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/top5.root.20260507.081152.575264
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/top5.root.20260507.081152.575264/output
Streaming final output from /tmp/top5.root.20260507.081152.575264/output...
"the"	6
"thy"	5
"to"	4
"s"	4
"world"	3
Removing temp directory /tmp/top5.root.20260507.081152.575264...


When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a tattered weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserved thy beauty's use,
If thou couldst answer This fair child of mine
Shall sum my count, and make my old excuse
Proving his beauty by succession thine.
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.

(A) Manual Python Approach

In [161]:
poem = """When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery so gazed on now,
Will be a tattered weed of small worth held:
Then being asked, where all thy beauty lies,
Where all the treasure of thy lusty days;
To say within thine own deep sunken eyes,
Were an all-eating shame, and thriftless praise.
How much more praise deserved thy beauty's use,
If thou couldst answer This fair child of mine
Shall sum my count, and make my old excuse
Proving his beauty by succession thine.
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold."""

words = re.findall(r'\w+', poem.lower())

# Mapper + Shuffle + Reducer combined
word_count = {}
for w in words:
    word_count[w] = word_count.get(w, 0) + 1

print(word_count)


{'when': 3, 'forty': 1, 'winters': 1, 'shall': 2, 'besiege': 1, 'thy': 7, 'brow': 1, 'and': 4, 'dig': 1, 'deep': 2, 'trenches': 1, 'in': 1, 'beauty': 4, 's': 3, 'field': 1, 'youth': 1, 'proud': 1, 'livery': 1, 'so': 1, 'gazed': 1, 'on': 1, 'now': 1, 'will': 1, 'be': 2, 'a': 1, 'tattered': 1, 'weed': 1, 'of': 3, 'small': 1, 'worth': 1, 'held': 1, 'then': 1, 'being': 1, 'asked': 1, 'where': 2, 'all': 3, 'lies': 1, 'the': 1, 'treasure': 1, 'lusty': 1, 'days': 1, 'to': 2, 'say': 1, 'within': 1, 'thine': 2, 'own': 1, 'sunken': 1, 'eyes': 1, 'were': 2, 'an': 1, 'eating': 1, 'shame': 1, 'thriftless': 1, 'praise': 2, 'how': 1, 'much': 1, 'more': 1, 'deserved': 1, 'use': 1, 'if': 1, 'thou': 3, 'couldst': 1, 'answer': 1, 'this': 2, 'fair': 1, 'child': 1, 'mine': 1, 'sum': 1, 'my': 2, 'count': 1, 'make': 1, 'old': 2, 'excuse': 1, 'proving': 1, 'his': 1, 'by': 1, 'succession': 1, 'new': 1, 'made': 1, 'art': 1, 'see': 1, 'blood': 1, 'warm': 1, 'feel': 1, 'st': 1, 'it': 1, 'cold': 1}


In [162]:
#Character Count
char_count = {}

for ch in poem.lower():
    if ch.isalnum():
        char_count[ch] = char_count.get(ch, 0) + 1

print(char_count)

{'w': 19, 'h': 34, 'e': 69, 'n': 28, 'f': 9, 'o': 28, 'r': 24, 't': 41, 'y': 21, 'i': 27, 's': 37, 'a': 32, 'l': 27, 'b': 11, 'g': 6, 'd': 24, 'p': 6, 'c': 9, 'u': 19, 'v': 3, 'z': 1, 'm': 11, 'k': 3, 'x': 1}


In [163]:
#Average Word Length
word_lengths = {}

for w in words:
    word_lengths.setdefault(w, []).append(len(w))

avg_word_len = {w: sum(v)/len(v) for w, v in word_lengths.items()}

print(avg_word_len)

{'when': 4.0, 'forty': 5.0, 'winters': 7.0, 'shall': 5.0, 'besiege': 7.0, 'thy': 3.0, 'brow': 4.0, 'and': 3.0, 'dig': 3.0, 'deep': 4.0, 'trenches': 8.0, 'in': 2.0, 'beauty': 6.0, 's': 1.0, 'field': 5.0, 'youth': 5.0, 'proud': 5.0, 'livery': 6.0, 'so': 2.0, 'gazed': 5.0, 'on': 2.0, 'now': 3.0, 'will': 4.0, 'be': 2.0, 'a': 1.0, 'tattered': 8.0, 'weed': 4.0, 'of': 2.0, 'small': 5.0, 'worth': 5.0, 'held': 4.0, 'then': 4.0, 'being': 5.0, 'asked': 5.0, 'where': 5.0, 'all': 3.0, 'lies': 4.0, 'the': 3.0, 'treasure': 8.0, 'lusty': 5.0, 'days': 4.0, 'to': 2.0, 'say': 3.0, 'within': 6.0, 'thine': 5.0, 'own': 3.0, 'sunken': 6.0, 'eyes': 4.0, 'were': 4.0, 'an': 2.0, 'eating': 6.0, 'shame': 5.0, 'thriftless': 10.0, 'praise': 6.0, 'how': 3.0, 'much': 4.0, 'more': 4.0, 'deserved': 8.0, 'use': 3.0, 'if': 2.0, 'thou': 4.0, 'couldst': 7.0, 'answer': 6.0, 'this': 4.0, 'fair': 4.0, 'child': 5.0, 'mine': 4.0, 'sum': 3.0, 'my': 2.0, 'count': 5.0, 'make': 4.0, 'old': 3.0, 'excuse': 6.0, 'proving': 7.0, 'his':

In [164]:
#Global Average Word Length
total_len = sum(len(w) for w in words)
total_count = len(words)

print("Global Average:", total_len / total_count)

Global Average: 4.083333333333333


In [165]:
#Top 5 Most Frequent Words
top5 = sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:5]
print(top5)

[('thy', 7), ('and', 4), ('beauty', 4), ('when', 3), ('s', 3)]


(B) MRJob Framework Approach

In [166]:
with open("Poem_2.txt", "w", encoding="utf-8") as f:
    f.write(poem)

In [167]:
#Word Count
%%file wordcount.py
from mrjob.job import MRJob
import re

class WordCount(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Overwriting wordcount.py


In [168]:
!python wordcount.py /content/Poem_2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260507.081153.482988
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260507.081153.482988/output
Streaming final output from /tmp/wordcount.root.20260507.081153.482988/output...
"a"	1
"all"	3
"an"	1
"and"	4
"answer"	1
"art"	1
"asked"	1
"be"	2
"beauty"	4
"being"	1
"besiege"	1
"blood"	1
"brow"	1
"by"	1
"child"	1
"cold"	1
"couldst"	1
"count"	1
"days"	1
"deep"	2
"old"	2
"on"	1
"own"	1
"praise"	2
"proud"	1
"proving"	1
"s"	3
"say"	1
"see"	1
"shall"	2
"shame"	1
"small"	1
"so"	1
"st"	1
"succession"	1
"sum"	1
"sunken"	1
"tattered"	1
"the"	1
"then"	1
"thine"	2
"this"	2
"thou"	3
"deserved"	1
"dig"	1
"eating"	1
"excuse"	1
"eyes"	1
"fair"	1
"feel"	1
"field"	1
"forty"	1
"gazed"	1
"held"	1
"his"	1
"how"	1
"if"	1
"in"	1
"it"	1
"lies"	1
"livery"	1
"lusty"	1
"made"	1
"make"	1
"mine"	1
"more"	1
"much"	1
"my"	2
"new"	1
"now"	1
"of"	3
"thriftless"	1
"thy"	7

In [169]:
#Character Count
%%file charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):

    def mapper(self, _, line):
        for ch in line.lower():
            if ch.isalnum():
                yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Overwriting charcount.py


In [170]:
!python charcount.py /content/Poem_2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charcount.root.20260507.081154.306605
Running step 1 of 1...
job output is in /tmp/charcount.root.20260507.081154.306605/output
Streaming final output from /tmp/charcount.root.20260507.081154.306605/output...
"a"	32
"b"	11
"c"	9
"d"	24
"e"	69
"o"	28
"p"	6
"r"	24
"s"	37
"t"	41
"f"	9
"g"	6
"h"	34
"i"	27
"k"	3
"l"	27
"m"	11
"n"	28
"u"	19
"v"	3
"w"	19
"x"	1
"y"	21
"z"	1
Removing temp directory /tmp/charcount.root.20260507.081154.306605...


In [171]:
#Average Word Length
%%file avg_word_len.py
from mrjob.job import MRJob
import re

class AvgWordLen(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, len(word)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals) / len(vals)

if __name__ == "__main__":
    AvgWordLen.run()

Overwriting avg_word_len.py


In [172]:
!python avg_word_len.py /content/Poem_2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_word_len.root.20260507.081155.067176
Running step 1 of 1...
job output is in /tmp/avg_word_len.root.20260507.081155.067176/output
Streaming final output from /tmp/avg_word_len.root.20260507.081155.067176/output...
"a"	1.0
"all"	3.0
"an"	2.0
"and"	3.0
"answer"	6.0
"art"	3.0
"asked"	5.0
"be"	2.0
"beauty"	6.0
"being"	5.0
"besiege"	7.0
"blood"	5.0
"brow"	4.0
"by"	2.0
"child"	5.0
"cold"	4.0
"couldst"	7.0
"count"	5.0
"days"	4.0
"deep"	4.0
"deserved"	8.0
"on"	2.0
"own"	3.0
"praise"	6.0
"proud"	5.0
"proving"	7.0
"s"	1.0
"say"	3.0
"see"	3.0
"shall"	5.0
"shame"	5.0
"small"	5.0
"so"	2.0
"st"	2.0
"succession"	10.0
"sum"	3.0
"sunken"	6.0
"tattered"	8.0
"the"	3.0
"then"	4.0
"thine"	5.0
"this"	4.0
"thou"	4.0
"dig"	3.0
"eating"	6.0
"excuse"	6.0
"eyes"	4.0
"fair"	4.0
"feel"	4.0
"field"	5.0
"forty"	5.0
"gazed"	5.0
"held"	4.0
"his"	3.0
"how"	3.0
"if"	2.0
"in"	2.0
"it"	2.0
"lies"	4.

In [173]:
#Global Average Word Length
%%file global_avg.py
from mrjob.job import MRJob
import re

class GlobalAvg(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield "all", (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0
        for l, c in values:
            total_len += l
            total_count += c
        yield "Global Average", total_len / total_count

if __name__ == "__main__":
    GlobalAvg.run()

Overwriting global_avg.py


In [174]:
!python global_avg.py Poem_2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/global_avg.root.20260507.081155.842748
Running step 1 of 1...
job output is in /tmp/global_avg.root.20260507.081155.842748/output
Streaming final output from /tmp/global_avg.root.20260507.081155.842748/output...
"Global Average"	4.083333333333333
Removing temp directory /tmp/global_avg.root.20260507.081155.842748...


In [175]:
#Top 5 Most Frequent Words
%%file top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

class Top5(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_count),
            MRStep(reducer=self.reducer_top5)
        ]

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, 1

    def reducer_count(self, key, values):
        yield None, (sum(values), key)

    def reducer_top5(self, _, values):
        for count, word in sorted(values, reverse=True)[:5]:
            yield word, count

if __name__ == "__main__":
    Top5.run()

Overwriting top5.py


In [176]:
!python top5.py Poem_2.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/top5.root.20260507.081156.729972
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/top5.root.20260507.081156.729972/output
Streaming final output from /tmp/top5.root.20260507.081156.729972/output...
"thy"	7
"beauty"	4
"and"	4
"when"	3
"thou"	3
Removing temp directory /tmp/top5.root.20260507.081156.729972...


  Look in thy glass and tell the face thou viewest,
  Now is the time that face should form another,
  Whose fresh repair if now thou not renewest,
  Thou dost beguile the world, unbless some mother.
  For where is she so fair whose uneared womb
  Disdains the tillage of thy husbandry?
  Or who is he so fond will be the tomb,
  Of his self-love to stop posterity?
  Thou art thy mother's glass and she in thee
  Calls back the lovely April of her prime,
  So thou through windows of thine age shalt see,
  Despite of wrinkles this thy golden time.
    But if thou live remembered not to be,
    Die single and thine image dies with thee.

(A) Manual Approach

In [177]:
import re

# INPUT
text = """Look in thy glass and tell the face thou viewest,
Now is the time that face should form another,
Whose fresh repair if now thou not renewest,
Thou dost beguile the world, unbless some mother.
For where is she so fair whose uneared womb
Disdains the tillage of thy husbandry?
Or who is he so fond will be the tomb,
Of his self-love to stop posterity?
Thou art thy mother's glass and she in thee
Calls back the lovely April of her prime,
So thou through windows of thine age shalt see,
Despite of wrinkles this thy golden time.
But if thou live remembered not to be,
Die single and thine image dies with thee."""

# Preprocessing
words = re.findall(r'\w+', text.lower())

# WORD COUNT
mapped = [(w, 1) for w in words]

shuffle = {}
for w, c in mapped:
    shuffle.setdefault(w, []).append(c)

word_count = {w: sum(v) for w, v in shuffle.items()}

print("\n Word Count")
print(word_count)

#  CHARACTER COUNT (ignore spaces)
mapped = []

for ch in text.lower():
    if ch.isalnum():
        mapped.append((ch, 1))

shuffle = {}
for ch, c in mapped:
    shuffle.setdefault(ch, []).append(c)

char_count = {ch: sum(v) for ch, v in shuffle.items()}

print("\n Character Count")
print(char_count)

# AVERAGE WORD LENGTH (PER WORD)
mapped = [(w, len(w)) for w in words]

shuffle = {}
for w, l in mapped:
    shuffle.setdefault(w, []).append(l)

avg_word_len = {w: sum(v)/len(v) for w, v in shuffle.items()}

print("\n Avg Word Length (Per Word)")
print(avg_word_len)

# GLOBAL AVERAGE WORD LENGTH
total_len = sum(len(w) for w in words)
total_count = len(words)

global_avg = total_len / total_count

print("\n Global Average Word Length")
print(global_avg)

# TOP 5 MOST FREQUENT WORDS
top5 = sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nTop 5 Words:")
print(top5)


 Word Count
{'look': 1, 'in': 2, 'thy': 4, 'glass': 2, 'and': 3, 'tell': 1, 'the': 6, 'face': 2, 'thou': 6, 'viewest': 1, 'now': 2, 'is': 3, 'time': 2, 'that': 1, 'should': 1, 'form': 1, 'another': 1, 'whose': 2, 'fresh': 1, 'repair': 1, 'if': 2, 'not': 2, 'renewest': 1, 'dost': 1, 'beguile': 1, 'world': 1, 'unbless': 1, 'some': 1, 'mother': 2, 'for': 1, 'where': 1, 'she': 2, 'so': 3, 'fair': 1, 'uneared': 1, 'womb': 1, 'disdains': 1, 'tillage': 1, 'of': 5, 'husbandry': 1, 'or': 1, 'who': 1, 'he': 1, 'fond': 1, 'will': 1, 'be': 2, 'tomb': 1, 'his': 1, 'self': 1, 'love': 1, 'to': 2, 'stop': 1, 'posterity': 1, 'art': 1, 's': 1, 'thee': 2, 'calls': 1, 'back': 1, 'lovely': 1, 'april': 1, 'her': 1, 'prime': 1, 'through': 1, 'windows': 1, 'thine': 2, 'age': 1, 'shalt': 1, 'see': 1, 'despite': 1, 'wrinkles': 1, 'this': 1, 'golden': 1, 'but': 1, 'live': 1, 'remembered': 1, 'die': 1, 'single': 1, 'image': 1, 'dies': 1, 'with': 1}

 Character Count
{'l': 25, 'o': 45, 'k': 3, 'i': 33, 'n': 22, '

(B)MRJob Framework Approach

In [178]:
poem = """  Look in thy glass and tell the face thou viewest,
  Now is the time that face should form another,
  Whose fresh repair if now thou not renewest,
  Thou dost beguile the world, unbless some mother.
  For where is she so fair whose uneared womb
  Disdains the tillage of thy husbandry?
  Or who is he so fond will be the tomb,
  Of his self-love to stop posterity?
  Thou art thy mother's glass and she in thee
  Calls back the lovely April of her prime,
  So thou through windows of thine age shalt see,
  Despite of wrinkles this thy golden time.
    But if thou live remembered not to be,
    Die single and thine image dies with thee."""

with open("Poem_3.txt", "w", encoding="utf-8") as f:
    f.write(poem)

In [179]:
#Word Count
%%file wordcount.py
from mrjob.job import MRJob
import re

class WordCount(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Overwriting wordcount.py


In [180]:
!python wordcount.py Poem_3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260507.081157.685970
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260507.081157.685970/output
Streaming final output from /tmp/wordcount.root.20260507.081157.685970/output...
"age"	1
"and"	3
"another"	1
"april"	1
"art"	1
"back"	1
"be"	2
"beguile"	1
"but"	1
"calls"	1
"despite"	1
"die"	1
"dies"	1
"disdains"	1
"dost"	1
"face"	2
"fair"	1
"fond"	1
"for"	1
"form"	1
"fresh"	1
"glass"	2
"golden"	1
"he"	1
"repair"	1
"s"	1
"see"	1
"self"	1
"shalt"	1
"she"	2
"should"	1
"single"	1
"so"	3
"some"	1
"stop"	1
"tell"	1
"that"	1
"the"	6
"thee"	2
"thine"	2
"this"	1
"thou"	6
"her"	1
"his"	1
"husbandry"	1
"if"	2
"image"	1
"in"	2
"is"	3
"live"	1
"look"	1
"love"	1
"lovely"	1
"mother"	2
"not"	2
"now"	2
"of"	5
"or"	1
"posterity"	1
"prime"	1
"remembered"	1
"renewest"	1
"through"	1
"thy"	4
"tillage"	1
"time"	2
"to"	2
"tomb"	1
"unbless"	1
"uneared"	1
"viewest"	

In [181]:
#Character Count
%%file charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):

    def mapper(self, _, line):
        for ch in line.lower():
            if ch.isalnum():
                yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Overwriting charcount.py


In [182]:
!python charcount.py Poem_3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charcount.root.20260507.081158.220776
Running step 1 of 1...
job output is in /tmp/charcount.root.20260507.081158.220776/output
Streaming final output from /tmp/charcount.root.20260507.081158.220776/output...
"a"	22
"b"	10
"c"	4
"d"	17
"e"	65
"f"	15
"n"	22
"o"	45
"p"	6
"r"	24
"s"	39
"g"	9
"h"	41
"i"	33
"k"	3
"l"	25
"m"	12
"t"	47
"u"	13
"v"	4
"w"	15
"y"	7
Removing temp directory /tmp/charcount.root.20260507.081158.220776...


In [183]:
#Average Word Length
%%file avg_word_len.py
from mrjob.job import MRJob
import re

class AvgWordLen(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, len(word)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals) / len(vals)

if __name__ == "__main__":
    AvgWordLen.run()

Overwriting avg_word_len.py


In [184]:
!python avg_word_len.py Poem_3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_word_len.root.20260507.081158.600994
Running step 1 of 1...
job output is in /tmp/avg_word_len.root.20260507.081158.600994/output
Streaming final output from /tmp/avg_word_len.root.20260507.081158.600994/output...
"age"	3.0
"and"	3.0
"another"	7.0
"april"	5.0
"art"	3.0
"back"	4.0
"be"	2.0
"beguile"	7.0
"but"	3.0
"calls"	5.0
"despite"	7.0
"die"	3.0
"dies"	4.0
"disdains"	8.0
"dost"	4.0
"face"	4.0
"fair"	4.0
"fond"	4.0
"for"	3.0
"form"	4.0
"fresh"	5.0
"glass"	5.0
"golden"	6.0
"he"	2.0
"repair"	6.0
"s"	1.0
"see"	3.0
"self"	4.0
"shalt"	5.0
"she"	3.0
"should"	6.0
"single"	6.0
"so"	2.0
"some"	4.0
"stop"	4.0
"tell"	4.0
"that"	4.0
"the"	3.0
"thee"	4.0
"thine"	5.0
"this"	4.0
"thou"	4.0
"her"	3.0
"his"	3.0
"husbandry"	9.0
"if"	2.0
"image"	5.0
"in"	2.0
"is"	2.0
"live"	4.0
"look"	4.0
"love"	4.0
"lovely"	6.0
"mother"	6.0
"not"	3.0
"now"	3.0
"of"	2.0
"or"	2.0
"posterity"	9.0
"p

In [185]:
#Global Average Word Length
%%file global_avg.py
from mrjob.job import MRJob
import re

class GlobalAvg(MRJob):

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield "all", (len(word), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0

        for l, c in values:
            total_len += l
            total_count += c

        yield "Global Average", total_len / total_count

if __name__ == "__main__":
    GlobalAvg.run()

Overwriting global_avg.py


In [186]:
!python global_avg.py Poem_3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/global_avg.root.20260507.081159.136615
Running step 1 of 1...
job output is in /tmp/global_avg.root.20260507.081159.136615/output
Streaming final output from /tmp/global_avg.root.20260507.081159.136615/output...
"Global Average"	4.085470085470085
Removing temp directory /tmp/global_avg.root.20260507.081159.136615...


In [187]:
#Top 5 Most Frequent Words
%%file top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

class Top5(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_count),
            MRStep(reducer=self.reducer_top5)
        ]

    def mapper(self, _, line):
        for word in re.findall(r'\w+', line.lower()):
            yield word, 1

    def reducer_count(self, key, values):
        yield None, (sum(values), key)

    def reducer_top5(self, _, values):
        for count, word in sorted(values, reverse=True)[:5]:
            yield word, count

if __name__ == "__main__":
    Top5.run()

Overwriting top5.py


In [188]:
!python top5.py Poem_3.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/top5.root.20260507.081159.645654
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/top5.root.20260507.081159.645654/output
Streaming final output from /tmp/top5.root.20260507.081159.645654/output...
"thou"	6
"the"	6
"of"	5
"thy"	4
"so"	3
Removing temp directory /tmp/top5.root.20260507.081159.645654...


Unthrifty loveliness why dost thou spend,
  Upon thy self thy beauty's legacy?
  Nature's bequest gives nothing but doth lend,
  And being frank she lends to those are free:
  Then beauteous niggard why dost thou abuse,
  The bounteous largess given thee to give?
  Profitless usurer why dost thou use
  So great a sum of sums yet canst not live?
  For having traffic with thy self alone,
  Thou of thy self thy sweet self dost deceive,
  Then how when nature calls thee to be gone,
  What acceptable audit canst thou leave?
    Thy unused beauty must be tombed with thee,
    Which used lives th' executor to be.

(A) Manual Python Approach

In [189]:
import re

# INPUT
text = """Unthrifty loveliness why dost thou spend,
Upon thy self thy beauty's legacy?
Nature's bequest gives nothing but doth lend,
And being frank she lends to those are free:
Then beauteous niggard why dost thou abuse,
The bounteous largess given thee to give?
Profitless usurer why dost thou use
So great a sum of sums yet canst not live?
For having traffic with thy self alone,
Thou of thy self thy sweet self dost deceive,
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
Thy unused beauty must be tombed with thee,
Which used lives th' executor to be."""

# Preprocessing
words = re.findall(r'\w+', text.lower())

#  WORD COUNT
mapped = [(w, 1) for w in words]

shuffle = {}
for w, c in mapped:
    shuffle.setdefault(w, []).append(c)

word_count = {w: sum(v) for w, v in shuffle.items()}

print("\nQ1: Word Count")
print(word_count)

#  CHARACTER COUNT
mapped = []

for ch in text.lower():
    if ch.isalnum():
        mapped.append((ch, 1))

shuffle = {}
for ch, c in mapped:
    shuffle.setdefault(ch, []).append(c)

char_count = {ch: sum(v) for ch, v in shuffle.items()}

print("\n Character Count")
print(char_count)

#  AVG WORD LENGTH (PER WORD)
mapped = [(w, len(w)) for w in words]

shuffle = {}
for w, l in mapped:
    shuffle.setdefault(w, []).append(l)

avg_word_len = {w: sum(v)/len(v) for w, v in shuffle.items()}

print("\n Avg Word Length")
print(avg_word_len)

#  GLOBAL AVG WORD LENGTH
total_len = sum(len(w) for w in words)
total_count = len(words)

print("\n Global Average:", total_len / total_count)

#  TOP 5 WORDS
top5 = sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nTop 5 Words:")
print(top5)


Q1: Word Count
{'unthrifty': 1, 'loveliness': 1, 'why': 3, 'dost': 4, 'thou': 5, 'spend': 1, 'upon': 1, 'thy': 6, 'self': 4, 'beauty': 2, 's': 2, 'legacy': 1, 'nature': 2, 'bequest': 1, 'gives': 1, 'nothing': 1, 'but': 1, 'doth': 1, 'lend': 1, 'and': 1, 'being': 1, 'frank': 1, 'she': 1, 'lends': 1, 'to': 4, 'those': 1, 'are': 1, 'free': 1, 'then': 2, 'beauteous': 1, 'niggard': 1, 'abuse': 1, 'the': 1, 'bounteous': 1, 'largess': 1, 'given': 1, 'thee': 3, 'give': 1, 'profitless': 1, 'usurer': 1, 'use': 1, 'so': 1, 'great': 1, 'a': 1, 'sum': 1, 'of': 2, 'sums': 1, 'yet': 1, 'canst': 2, 'not': 1, 'live': 1, 'for': 1, 'having': 1, 'traffic': 1, 'with': 2, 'alone': 1, 'sweet': 1, 'deceive': 1, 'how': 1, 'when': 1, 'calls': 1, 'be': 3, 'gone': 1, 'what': 1, 'acceptable': 1, 'audit': 1, 'leave': 1, 'unused': 1, 'must': 1, 'tombed': 1, 'which': 1, 'used': 1, 'lives': 1, 'th': 1, 'executor': 1}

 Character Count
{'u': 29, 'n': 26, 't': 55, 'h': 34, 'r': 15, 'i': 18, 'f': 13, 'y': 14, 'l': 18, '

(B) MRJob Framework Approach

In [190]:
poem = """Unthrifty loveliness why dost thou spend,
Upon thy self thy beauty's legacy?
Nature's bequest gives nothing but doth lend,
And being frank she lends to those are free:
Then beauteous niggard why dost thou abuse,
The bounteous largess given thee to give?
Profitless usurer why dost thou use
So great a sum of sums yet canst not live?
For having traffic with thy self alone,
Thou of thy self thy sweet self dost deceive,
Then how when nature calls thee to be gone,
What acceptable audit canst thou leave?
Thy unused beauty must be tombed with thee,
Which used lives th' executor to be."""

with open("Poem_4.txt", "w", encoding="utf-8") as f:
    f.write(poem)

In [191]:
#Word Count
%%file wordcount.py
from mrjob.job import MRJob
import re

class WordCount(MRJob):
    def mapper(self, _, line):
        for w in re.findall(r'\w+', line.lower()):
            yield w, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()

Overwriting wordcount.py


In [192]:
!python wordcount.py Poem_4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260507.081200.125768
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260507.081200.125768/output
Streaming final output from /tmp/wordcount.root.20260507.081200.125768/output...
"a"	1
"abuse"	1
"acceptable"	1
"alone"	1
"and"	1
"are"	1
"audit"	1
"be"	3
"beauteous"	1
"beauty"	2
"being"	1
"bequest"	1
"bounteous"	1
"but"	1
"calls"	1
"canst"	2
"deceive"	1
"dost"	4
"doth"	1
"of"	2
"profitless"	1
"s"	2
"self"	4
"she"	1
"so"	1
"spend"	1
"sum"	1
"sums"	1
"sweet"	1
"th"	1
"the"	1
"thee"	3
"then"	2
"those"	1
"thou"	5
"thy"	6
"executor"	1
"for"	1
"frank"	1
"free"	1
"give"	1
"given"	1
"gives"	1
"gone"	1
"great"	1
"having"	1
"how"	1
"largess"	1
"leave"	1
"legacy"	1
"lend"	1
"lends"	1
"live"	1
"lives"	1
"loveliness"	1
"must"	1
"nature"	2
"niggard"	1
"not"	1
"nothing"	1
"to"	4
"tombed"	1
"traffic"	1
"unthrifty"	1
"unused"	1
"upon"	1
"use"	1
"used"	1
"u

In [193]:
#Character Count
%%file charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):
    def mapper(self, _, line):
        for ch in line.lower():
            if ch.isalnum():
                yield ch, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Overwriting charcount.py


In [194]:
!python charcount.py Poem_4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charcount.root.20260507.081200.527867
Running step 1 of 1...
job output is in /tmp/charcount.root.20260507.081200.527867/output
Streaming final output from /tmp/charcount.root.20260507.081200.527867/output...
"a"	25
"b"	13
"c"	10
"d"	15
"e"	66
"o"	32
"p"	4
"q"	1
"r"	15
"s"	39
"t"	55
"f"	13
"g"	12
"h"	34
"i"	18
"k"	1
"l"	18
"m"	4
"n"	26
"u"	29
"v"	9
"w"	10
"x"	1
"y"	14
Removing temp directory /tmp/charcount.root.20260507.081200.527867...


In [195]:
#Avg Word Length
%%file avg_word_len.py
from mrjob.job import MRJob
import re

class AvgWordLen(MRJob):
    def mapper(self, _, line):
        for w in re.findall(r'\w+', line.lower()):
            yield w, len(w)

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals)/len(vals)

if __name__ == "__main__":
    AvgWordLen.run()

Overwriting avg_word_len.py


In [196]:
!python avg_word_len.py Poem_4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_word_len.root.20260507.081200.966451
Running step 1 of 1...
job output is in /tmp/avg_word_len.root.20260507.081200.966451/output
Streaming final output from /tmp/avg_word_len.root.20260507.081200.966451/output...
"a"	1.0
"abuse"	5.0
"acceptable"	10.0
"alone"	5.0
"and"	3.0
"are"	3.0
"audit"	5.0
"be"	2.0
"beauteous"	9.0
"beauty"	6.0
"being"	5.0
"bequest"	7.0
"bounteous"	9.0
"but"	3.0
"calls"	5.0
"canst"	5.0
"deceive"	7.0
"dost"	4.0
"doth"	4.0
"of"	2.0
"profitless"	10.0
"s"	1.0
"self"	4.0
"she"	3.0
"so"	2.0
"spend"	5.0
"sum"	3.0
"sums"	4.0
"sweet"	5.0
"th"	2.0
"the"	3.0
"thee"	4.0
"then"	4.0
"those"	5.0
"thou"	4.0
"thy"	3.0
"executor"	8.0
"for"	3.0
"frank"	5.0
"free"	4.0
"give"	4.0
"given"	5.0
"gives"	5.0
"gone"	4.0
"great"	5.0
"having"	6.0
"how"	3.0
"largess"	7.0
"leave"	5.0
"legacy"	6.0
"lend"	4.0
"lends"	5.0
"live"	4.0
"lives"	5.0
"loveliness"	10.0
"must"	4.0
"n

In [197]:
#Global Avg
%%file global_avg.py
from mrjob.job import MRJob
import re

class GlobalAvg(MRJob):
    def mapper(self, _, line):
        for w in re.findall(r'\w+', line.lower()):
            yield "all", (len(w), 1)

    def reducer(self, key, values):
        total_len = 0
        total_count = 0
        for l, c in values:
            total_len += l
            total_count += c
        yield "Global Average", total_len / total_count

if __name__ == "__main__":
    GlobalAvg.run()

Overwriting global_avg.py


In [198]:
!python global_avg.py Poem_4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/global_avg.root.20260507.081201.380836
Running step 1 of 1...
job output is in /tmp/global_avg.root.20260507.081201.380836/output
Streaming final output from /tmp/global_avg.root.20260507.081201.380836/output...
"Global Average"	4.377358490566038
Removing temp directory /tmp/global_avg.root.20260507.081201.380836...


In [199]:
#Top 5 Words
%%file top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep
import re

class Top5(MRJob):

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_count),
            MRStep(reducer=self.reducer_top5)
        ]

    def mapper(self, _, line):
        for w in re.findall(r'\w+', line.lower()):
            yield w, 1

    def reducer_count(self, key, values):
        yield None, (sum(values), key)

    def reducer_top5(self, _, values):
        for count, word in sorted(values, reverse=True)[:5]:
            yield word, count

if __name__ == "__main__":
    Top5.run()

Overwriting top5.py


In [200]:
!python top5.py Poem_4.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/top5.root.20260507.081201.708313
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/top5.root.20260507.081201.708313/output
Streaming final output from /tmp/top5.root.20260507.081201.708313/output...
"thy"	6
"thou"	5
"to"	4
"self"	4
"dost"	4
Removing temp directory /tmp/top5.root.20260507.081201.708313...


Q6 Compute average marks for each student.

Input:
A 80
B 70
A 90
B 60
A 100

(A) Manual Python Approach

In [201]:
# INPUT
data = [
    "A 80",
    "B 70",
    "A 90",
    "B 60",
    "A 100"
]

# MAPPER
mapped = []

for line in data:
    student, marks = line.split()
    mapped.append((student, int(marks)))

print("Mapper Output:")
print(mapped)

# SHUFFLE
shuffle = {}

for student, marks in mapped:
    shuffle.setdefault(student, []).append(marks)

print("\nShuffle Output:")
print(shuffle)

# REDUCER (Average)
avg_marks = {}

for student, marks_list in shuffle.items():
    avg_marks[student] = sum(marks_list) / len(marks_list)

print("\nFinal Average Marks:")
for student, avg in avg_marks.items():
    print(student, ":", avg)

Mapper Output:
[('A', 80), ('B', 70), ('A', 90), ('B', 60), ('A', 100)]

Shuffle Output:
{'A': [80, 90, 100], 'B': [70, 60]}

Final Average Marks:
A : 90.0
B : 65.0


In [202]:
%%writefile q6_avg_marks_mrjob.py
from mrjob.job import MRJob

class AvgMarksMR(MRJob):
    def mapper(self, _, line):
        student, marks = line.split()
        yield student, int(marks)

    def reducer(self, student, marks):
        marks = list(marks)
        yield student, sum(marks) / len(marks)

if __name__ == "__main__":
    AvgMarksMR.run()

Overwriting q6_avg_marks_mrjob.py


In [203]:
with open("q6_input.txt", "w") as f:
    f.write("A 80\nB 70\nA 90\nB 60\nA 100\n")

!python q6_avg_marks_mrjob.py q6_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q6_avg_marks_mrjob.root.20260507.081202.148093
Running step 1 of 1...
job output is in /tmp/q6_avg_marks_mrjob.root.20260507.081202.148093/output
Streaming final output from /tmp/q6_avg_marks_mrjob.root.20260507.081202.148093/output...
"A"	90.0
"B"	65.0
Removing temp directory /tmp/q6_avg_marks_mrjob.root.20260507.081202.148093...


Q7 Compute average salary per department and Highest Paid Department (Based on Average
Salary)

Input:

HR 50000

IT 70000

HR 60000

IT 80000

In [204]:
#INPUT
data = [
    "HR 50000",
    "IT 70000",
    "HR 60000",
    "IT 80000"
]

# MAPPER
mapped = []

for line in data:
    dept, salary = line.split()
    mapped.append((dept, int(salary)))

print("Mapper Output:")
print(mapped)

#SHUFFLE
shuffle = {}

for dept, salary in mapped:
    shuffle.setdefault(dept, []).append(salary)

print("\nShuffle Output:")
print(shuffle)

# REDUCER (Average)
avg_salary = {}

for dept, salaries in shuffle.items():
    avg_salary[dept] = sum(salaries) / len(salaries)

print("\nAverage Salary per Department:")
print(avg_salary)

#HIGHEST PAID DEPARTMENT
highest_dept = max(avg_salary, key=avg_salary.get)

print("\nHighest Paid Department:")

Mapper Output:
[('HR', 50000), ('IT', 70000), ('HR', 60000), ('IT', 80000)]

Shuffle Output:
{'HR': [50000, 60000], 'IT': [70000, 80000]}

Average Salary per Department:
{'HR': 55000.0, 'IT': 75000.0}

Highest Paid Department:


In [205]:
%%writefile q7_avg_salary_mrjob.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class AvgSalaryAndHighestDeptMR(MRJob):
    def steps(self):
        return [
            MRStep(mapper=self.mapper_salary, reducer=self.reducer_avg_salary),
            MRStep(mapper=self.mapper_find_max, reducer=self.reducer_find_max)
        ]

    def mapper_salary(self, _, line):
        dept, salary = line.split()
        yield dept, int(salary)

    def reducer_avg_salary(self, dept, salaries):
        salaries = list(salaries)
        avg_salary = sum(salaries) / len(salaries)
        yield dept, avg_salary

    def mapper_find_max(self, dept, avg_salary):
        yield "highest_paid_department", (avg_salary, dept)

    def reducer_find_max(self, key, values):
        best = max(values)
        yield key, {"department": best[1], "average_salary": best[0]}

if __name__ == "__main__":
    AvgSalaryAndHighestDeptMR.run()

Overwriting q7_avg_salary_mrjob.py


In [206]:
with open("q7_input.txt", "w") as f:
    f.write("HR 50000\nIT 70000\nHR 60000\nIT 80000\n")

!python q7_avg_salary_mrjob.py q7_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q7_avg_salary_mrjob.root.20260507.081202.688049
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/q7_avg_salary_mrjob.root.20260507.081202.688049/output
Streaming final output from /tmp/q7_avg_salary_mrjob.root.20260507.081202.688049/output...
"highest_paid_department"	{"department": "IT", "average_salary": 75000.0}
Removing temp directory /tmp/q7_avg_salary_mrjob.root.20260507.081202.688049...


Q8 Computer average temperature per country

New York,38

London,29

Tokyo,35

New York,32

Delhi,45

Ambala,35


(A) Manual Python Approach

In [207]:
data = [
    "New York,38",
    "London,29",
    "Tokyo,35",
    "New York,32",
    "Delhi,45",
    "Ambala,35"
]

mapped = []

for line in data:
    city, temp = line.split(",")
    mapped.append((city, int(temp)))

print("Mapper Output:")
print(mapped)

shuffle = {}

for city, temp in mapped:
    shuffle.setdefault(city, []).append(temp)

print("\nShuffle Output:")
print(shuffle)

avg_temp = {}

for city, temps in shuffle.items():
    avg_temp[city] = sum(temps) / len(temps)

print("\nAverage Temperature per City:")
for city, avg in avg_temp.items():
    print(city, ":", avg)

Mapper Output:
[('New York', 38), ('London', 29), ('Tokyo', 35), ('New York', 32), ('Delhi', 45), ('Ambala', 35)]

Shuffle Output:
{'New York': [38, 32], 'London': [29], 'Tokyo': [35], 'Delhi': [45], 'Ambala': [35]}

Average Temperature per City:
New York : 35.0
London : 29.0
Tokyo : 35.0
Delhi : 45.0
Ambala : 35.0


(B) MrJob Approach

In [208]:
%%writefile q8_avg_temperature_mrjob.py
from mrjob.job import MRJob

class AvgTemperatureMR(MRJob):
    def mapper(self, _, line):
        place, temp = line.split(",")
        yield place.strip(), float(temp)

    def reducer(self, place, temps):
        temps = list(temps)
        yield place, sum(temps) / len(temps)

if __name__ == "__main__":
    AvgTemperatureMR.run()

Overwriting q8_avg_temperature_mrjob.py


In [209]:
with open("q8_input.txt", "w") as f:
    f.write("New York,38\nLondon,29\nTokyo,35\nNew York,32\nDelhi,45\nAmbala,35\n")

!python q8_avg_temperature_mrjob.py q8_input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/q8_avg_temperature_mrjob.root.20260507.081203.250867
Running step 1 of 1...
job output is in /tmp/q8_avg_temperature_mrjob.root.20260507.081203.250867/output
Streaming final output from /tmp/q8_avg_temperature_mrjob.root.20260507.081203.250867/output...
"Ambala"	35.0
"Delhi"	45.0
"Tokyo"	35.0
"London"	29.0
"New York"	35.0
Removing temp directory /tmp/q8_avg_temperature_mrjob.root.20260507.081203.250867...


Q9 Redo Q8 with the dataset available on:
https://www.kaggle.com/datasets/heemalichaudhari/global-land-temperatures

(A) Manual Python Approach

In [210]:
import csv

file_path = "/content/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv"

mapped = []

with open(file_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['AverageTemperature'] != '':
            country = row['Country']
            temp = float(row['AverageTemperature'])
            mapped.append((country, temp))

shuffle = {}

for country, temp in mapped:
    shuffle.setdefault(country, []).append(temp)

avg_temp = {}

for country, temps in shuffle.items():
    avg_temp[country] = sum(temps) / len(temps)

print("Average Temperature per Country (Sample):")
for i, (country, avg) in enumerate(avg_temp.items()):
    print(country, ":", round(avg, 2))
    if i == 9:
        break

Average Temperature per Country (Sample):
Côte D'Ivoire : 26.16
Ethiopia : 17.53
India : 25.81
Syria : 17.37
Egypt : 20.9
Turkey : 13.79
Iraq : 22.61
Thailand : 27.16
Brazil : 22.85
Germany : 8.92


(B) MrJob Approach

In [211]:
%%file avg_temp_country.py
from mrjob.job import MRJob
import csv

class AvgTempCountry(MRJob):

    def mapper(self, _, line):
        # Skip header
        if line.startswith("dt"):
            return

        try:
            row = list(csv.reader([line]))[0]
            temp = row[1]
            country = row[3]

            if temp != '':
                yield country, float(temp)

        except:
            pass

    def reducer(self, key, values):
        vals = list(values)
        yield key, sum(vals) / len(vals)

if __name__ == "__main__":
    AvgTempCountry.run()

Overwriting avg_temp_country.py


In [212]:
!python avg_temp_country.py /content/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_temp_country.root.20260507.081205.274014
Running step 1 of 1...
job output is in /tmp/avg_temp_country.root.20260507.081205.274014/output
Streaming final output from /tmp/avg_temp_country.root.20260507.081205.274014/output...
"Abidjan"	26.16373719752392
"Addis Abeba"	17.52507266229899
"Ahmadabad"	26.529852941176472
"Aleppo"	17.37058733360226
"Alexandria"	20.312617029257314
"Ankara"	10.39211714770798
"Baghdad"	22.61434568965517
"Bangalore"	24.855895933014352
"Bangkok"	27.164733303650937
"Belo Horizonte"	21.071396469465647
"Berlin"	8.916233733417561
"Bogot\u00e1"	20.002264618434094
"Bombay"	26.631452153110047
"Bras\u00edlia"	21.72759494274809
"Cairo"	21.221259213759215
"Calcutta"	26.04215205371248
"Cali"	21.79702720403023
"Cape Town"	16.079079551521623
"Casablanca"	17.184157858613588
"Changchun"	4.923797927461139
"Chengdu"	10.638042143838755
"Chicago"	10.0706437440